In [ ]:
```json
{
 "cells": [
  {
   "cell_type": "markdown",
   "source": [
    "# ODI to Databricks Migration - WC_BADGE_EVENTS_F\n",
    "\n",
    "**Conversion Timestamp:** 2024-07-30T12:00:00Z\n",
    "\n",
    "This notebook contains the Spark SQL conversion of the ODI session for `WC_BADGE_EVENTS_F`.\n",
    "It handles staging, flow, error logging, PK violation detection, and MERGE operations into the target table.\n",
    "\n",
    "**Inferred DDL Assumptions:**\n",
    "- `workspace.prxbi_dw.wc_etl_parameters` table is assumed to exist with `etl_job_type`, `etl_last_extract_time`, `etl_current_extract_time`, and `row_wid` columns.\n",
    "- `workspace.prxbi_dw.wc_mercury_badgeevent_f` target table's `row_wid` column is `GENERATED ALWAYS AS IDENTITY`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# COMMAND ----------\n",
    "# Create Databricks Widgets for ETL Parameters\n",
    "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"ETL Job Type\")\n",
    "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\", \"Datasource Number ID\")\n",
    "dbutils.widgets.text(\"ETL_PROC_WID\", \"\", \"ETL Process Widget ID\")\n",
    "dbutils.widgets.text(\"ODI_SESS_NO\", \"\", \"ODI Session Number\")"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# ETL Parameters\n",
    "\n",
    "The following cells define and display the ETL parameters used throughout this notebook. These parameters are typically set by an orchestrator like Databricks Workflows or by manually running the notebook."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Inferred DDL for wc_etl_parameters table\n",
    "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_etl_parameters (\n",
    "    etl_job_type                STRING,\n",
    "    etl_last_extract_time       TIMESTAMP,\n",
    "    etl_current_extract_time    TIMESTAMP,\n",
    "    row_wid                     BIGINT\n",
    ") USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {2}: Get ETL last extract time\n",
    "CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS\n",
    "SELECT etl_last_extract_time FROM workspace.prxbi_dw.wc_etl_parameters WHERE etl_job_type = '${ETL_JOB_TYPE}';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {3}: Get ETL current extract time\n",
    "CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS\n",
    "SELECT etl_current_extract_time FROM workspace.prxbi_dw.wc_etl_parameters WHERE etl_job_type = '${ETL_JOB_TYPE}';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {6}: Get ETL ROW_WID parameter\n",
    "CREATE OR REPLACE TEMPORARY VIEW v_row_wid_param AS\n",
    "SELECT row_wid FROM workspace.prxbi_dw.wc_etl_parameters WHERE etl_job_type = '${ETL_JOB_TYPE}';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# COMMAND ----------\n",
    "# Display current ETL parameter values\n",
    "print(f\"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}\")\n",
    "print(f\"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}\")\n",
    "print(f\"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}\")\n",
    "print(f\"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}\")\n",
    "display(spark.sql(\"SELECT etl_last_extract_time FROM v_etl_last_extract_time\"))\n",
    "display(spark.sql(\"SELECT etl_current_extract_time FROM v_etl_current_extract_time\"))\n",
    "display(spark.sql(\"SELECT row_wid FROM v_row_wid_param\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Staging Table\n",
    "\n",
    "This section handles the creation and population of the temporary staging table (`C$_`).\n",
    "The ODI MAX-based self-join deduplication pattern (F.12) is replaced with `ROW_NUMBER()` for correctness in Spark SQL."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {30}: Drop staging table\n",
    "DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_event_stg;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {40}: Create staging table\n",
    "CREATE TABLE workspace.prxbi_dw.c_badge_event_stg\n",
    "(\n",
    "    EVENTEDITIONGBSCODE STRING NULL,\n",
    "    EVENTTYPE           STRING NULL,\n",
    "    BADGEID             STRING NULL,\n",
    "    SOURCE              STRING NULL,\
    "    PRODUCTCODE         STRING NULL,\
    "    CUSTOMERTYPE        STRING NULL,\
    "    EVENTDATE           STRING NULL,\
    "    CREATEDDATE         STRING NULL\n",
    ")\n",
    "USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {50}: Insert into staging table (F.12 ODI MAX self-join dedup replaced with ROW_NUMBER)\n",
    "INSERT INTO workspace.prxbi_dw.c_badge_event_stg\n",
    "SELECT\n",
    "    EVENTEDITIONGBSCODE,\n",
    "    EVENTTYPE,\n",
    "    BADGEID,\n",
    "    SOURCE,\n",
    "    PRODUCTCODE,\
    "    CUSTOMERTYPE,\
    "    EVENTDATE,\
    "    CREATEDDATE\n",
    "FROM (\n",
    "    SELECT\n",
    "        AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,\n",
    "        AMERCURY_BADGEEVENT_TS.EVENTTYPE,\n",
    "        AMERCURY_BADGEEVENT_TS.BADGEID,\n",
    "        AMERCURY_BADGEEVENT_TS.SOURCE,\n",
    "        AMERCURY_BADGEEVENT_TS.PRODUCTCODE,\
    "        AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE,\
    "        AMERCURY_BADGEEVENT_TS.EVENTDATE,\
    "        AMERCURY_BADGEEVENT_TS.CREATEDDATE,\
    "        ROW_NUMBER() OVER (\n",
    "            PARTITION BY\n",
    "                AMERCURY_BADGEEVENT_TS.EVENTEDITIONGBSCODE,\n",
    "                AMERCURY_BADGEEVENT_TS.EVENTTYPE,\n",
    "                AMERCURY_BADGEEVENT_TS.BADGEID,\n",
    "                AMERCURY_BADGEEVENT_TS.SOURCE,\n",
    "                AMERCURY_BADGEEVENT_TS.PRODUCTCODE,\
    "                AMERCURY_BADGEEVENT_TS.CUSTOMERTYPE\n",
    "            ORDER BY\n",
    "                AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE DESC\n",
    "        ) AS rn\n",
    "    FROM\n",
    "        workspace.prxbi_ts.wc_mercury_badgeevent_ts AS AMERCURY_BADGEEVENT_TS\n",
    "    WHERE\n",
    "        AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)\n",
    "        AND AMERCURY_BADGEEVENT_TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)\n",
    ") AS filtered_source\n",
    "WHERE rn = 1;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Staging table record count\n",
    "SELECT COUNT(*) AS c_badge_event_stg_count FROM workspace.prxbi_dw.c_badge_event_stg;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {60}: Optimize staging table (replaces DBMS_STATS)\n",
    "OPTIMIZE workspace.prxbi_dw.c_badge_event_stg;"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Flow Table\n",
    "\n",
    "This section creates and populates the integration flow table (`I$_`) with data from the staging table, performing necessary lookups and transformations."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {80}: Drop flow table\n",
    "DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {90}: Create flow table\n",
    "CREATE TABLE workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow\n",
    "(\n",
    "    ROW_WID             BIGINT NULL,\n",
    "    INTEGRATION_ID      STRING NULL,\n",
    "    ETL_PROC_WID        BIGINT NULL,\
    "    EVENTEDITIONGBSCODE STRING NULL,\
    "    EVENTTYPE           STRING NULL,\
    "    BADGEID             STRING NULL,\
    "    SOURCE              STRING NULL,\
    "    PRODUCTCODE         STRING NULL,\
    "    CUSTOMERTYPE        STRING NULL,\
    "    EVENTDATE           STRING NULL,\
    "    CREATEDDATE         STRING NULL,\
    "    BADGE_WID           BIGINT NULL,\
    "    EVENT_EDITION_WID   BIGINT NULL,\
    "    OBU_WID             BIGINT NULL,\
    "    PRODUCT_WID         BIGINT NULL,\
    "    W_UPDATE_DT         TIMESTAMP NULL,\
    "    W_INSERT_DT         TIMESTAMP NULL,\
    "    EVENT_WID           BIGINT NULL,\
    "    IND_UPDATE          STRING\n",
    ")\n",
    "USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {100}: Insert into flow table with dimension lookups\n",
    "INSERT INTO workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow\n",
    "(\n",
    "    INTEGRATION_ID,\n",
    "    EVENTEDITIONGBSCODE,\n",
    "    EVENTTYPE,\n",
    "    BADGEID,\n",
    "    SOURCE,\n",
    "    PRODUCTCODE,\n",
    "    CUSTOMERTYPE,\n",
    "    EVENTDATE,\n",
    "    CREATEDDATE,\n",
    "    BADGE_WID,\n",
    "    EVENT_EDITION_WID,\n",
    "    OBU_WID,\n",
    "    PRODUCT_WID,\n",
    "    EVENT_WID,\n",
    "    IND_UPDATE\n",
    ")\n",
    "SELECT\n",
    "    CONCAT_WS('~',\n",
    "        FILTER2_A_1.EVENTEDITIONGBSCODE,\n",
    "        FILTER2_A_1.EVENTTYPE,\n",
    "        FILTER2_A_1.BADGEID,\n",
    "        FILTER2_A_1.SOURCE,\n",
    "        FILTER2_A_1.PRODUCTCODE,\
    "        FILTER2_A_1.CUSTOMERTYPE\n",
    "    ) AS INTEGRATION_ID,\n",
    "    FILTER2_A_1.EVENTEDITIONGBSCODE,\n",
    "    FILTER2_A_1.EVENTTYPE,\n",
    "    FILTER2_A_1.BADGEID,\n",
    "    FILTER2_A_1.SOURCE,\n",
    "    FILTER2_A_1.PRODUCTCODE,\
    "    FILTER2_A_1.CUSTOMERTYPE,\
    "    FILTER2_A_1.EVENTDATE,\
    "    FILTER2_A_1.CREATEDDATE,\
    "    COALESCE(WC_BADGE_DETAILS_D.ROW_WID, 0) AS BADGE_WID,\n",
    "    COALESCE(WC_EVENT_ED_D_1.ROW_WID, 0) AS EVENT_EDITION_WID,\n",
    "    COALESCE(WC_EVENT_ED_D_1.OBU_WID, 0) AS OBU_WID,\n",
    "    COALESCE(WC_BADGE_PRODUCT_D_SQ_BADGEP_1.ROW_WID_1, 0) AS PRODUCT_WID,\
    "    COALESCE(WC_EVENT_D.ROW_WID, 0) AS EVENT_WID,\
    "    'I' AS IND_UPDATE\n",
    "FROM\n",
    "    workspace.prxbi_dw.c_badge_event_stg AS FILTER2_A_1\n",
    "LEFT OUTER JOIN\n",
    "    workspace.prxbi_dw.wc_event_ed_d AS WC_EVENT_ED_D_1\n",
    "    ON FILTER2_A_1.EVENTEDITIONGBSCODE = CONCAT(rpad(WC_EVENT_ED_D_1.EVENT_ALPHA_CODE, 5, '-'), WC_EVENT_ED_D_1.EVENT_EDITION_CODE)\n",
    "LEFT OUTER JOIN\n",
    "    workspace.prxbi_dw.wc_badge_details_d AS WC_BADGE_DETAILS_D\n",
    "    ON FILTER2_A_1.BADGEID = WC_BADGE_DETAILS_D.BADGE_ID\n",
    "LEFT OUTER JOIN\n",
    "    (\n",
    "        SELECT\n",
    "            MAX(WC_BADGE_PRODUCT_D_SQ_BADGEPRO.ROW_WID) AS ROW_WID_1,\n",
    "            WC_BADGE_PRODUCT_D_SQ_BADGEPRO.SKU AS SKU_1\n",
    "        FROM\n",
    "            workspace.prxbi_dw.wc_badge_product_d AS WC_BADGE_PRODUCT_D_SQ_BADGEPRO\n",
    "        GROUP BY\n",
    "            WC_BADGE_PRODUCT_D_SQ_BADGEPRO.SKU\n",
    "    ) AS WC_BADGE_PRODUCT_D_SQ_BADGEP_1\n",
    "    ON WC_BADGE_PRODUCT_D_SQ_BADGEP_1.SKU_1 = FILTER2_A_1.PRODUCTCODE\n",
    "LEFT OUTER JOIN\n",
    "    workspace.prxbi_dw.wc_event_d AS WC_EVENT_D\n",
    "    ON WC_EVENT_ED_D_1.EVENT_INTEGRATION_ID = WC_EVENT_D.INTEGRATION_ID;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Flow table record count\n",
    "SELECT COUNT(*) AS i_flow_table_count FROM workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {110}: Create index on flow table (converted to OPTIMIZE ZORDER BY)\n",
    "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
    "OPTIMIZE workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow ZORDER BY (integration_id);"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {120}: Gather flow table stats (converted to OPTIMIZE)\n",
    "OPTIMIZE workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow;"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Error / Check Tables\n",
    "\n",
    "This section sets up and clears error logging and check tables (`E$_`, `SNP_CHECK_TAB`)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {130}: Create SNP_CHECK_TAB\n",
    "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.snp_check_tab\n",
    "(\n",
    "    CATALOG_NAME    STRING NULL,\n",
    "    SCHEMA_NAME     STRING NULL,\n",
    "    RESOURCE_NAME   STRING NULL,\n",
    "    FULL_RES_NAME   STRING NULL,\n",
    "    ERR_TYPE        STRING NULL,\n",
    "    ERR_MESS        STRING NULL,\n",
    "    CHECK_DATE      TIMESTAMP NULL,\n",
    "    ORIGIN          STRING NULL,\
    "    CONS_NAME       STRING NULL,\
    "    CONS_TYPE       STRING NULL,\
    "    ERR_COUNT       BIGINT NULL\n",
    ")\n",
    "USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {140}: Delete from SNP_CHECK_TAB for current origin\n",
    "DELETE FROM workspace.prxbi_dw.snp_check_tab\n",
    "WHERE\n",
    "    SCHEMA_NAME = 'prxbi_dw'\n",
    "    AND ORIGIN = '(76564095)Sales and Marketing.WC_BADGE_EVENTS_F'\n",
    "    AND ERR_TYPE = 'F';"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {150}: Create E$ error table\n",
    "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_wc_mercury_badgeevent_f\n",
    "(\n",
    "    ODI_ROW_ID          STRING NULL,\n",
    "    ODI_ERR_TYPE        STRING NULL,\n",
    "    ODI_ERR_MESS        STRING NULL,\
    "    ODI_CHECK_DATE      TIMESTAMP NULL,\n",
    "    ROW_WID             BIGINT NULL,\
    "    INTEGRATION_ID      STRING NULL,\
    "    ETL_PROC_WID        BIGINT NULL,\
    "    EVENTEDITIONGBSCODE STRING NULL,\
    "    EVENTTYPE           STRING NULL,\
    "    BADGEID             STRING NULL,\
    "    SOURCE              STRING NULL,\
    "    PRODUCTCODE         STRING NULL,\
    "    CUSTOMERTYPE        STRING NULL,\
    "    EVENTDATE           STRING NULL,\
    "    CREATEDDATE         STRING NULL,\
    "    BADGE_WID           BIGINT NULL,\
    "    EVENT_EDITION_WID   BIGINT NULL,\
    "    OBU_WID             BIGINT NULL,\
    "    PRODUCT_WID         BIGINT NULL,\
    "    W_UPDATE_DT         TIMESTAMP NULL,\
    "    W_INSERT_DT         TIMESTAMP NULL,\
    "    EVENT_WID           BIGINT NULL,\
    "    ODI_ORIGIN          STRING NULL,\
    "    ODI_CONS_NAME       STRING NULL,\
    "    ODI_CONS_TYPE       STRING NULL,\
    "    ODI_PK              STRING NULL,\n",
    "    ODI_SESS_NO         STRING NULL\n",
    ")\n",
    "USING DELTA;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {160}: Delete from E$ error table for current session\n",
    "DELETE FROM workspace.prxbi_dw.e_wc_mercury_badgeevent_f\n",
    "WHERE\n",
    "    (ODI_ERR_TYPE = 'S' AND 1 = 0) -- 'F' = 'S' converted to 1 = 0\n",
    "    OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(76564095)Sales and Marketing.WC_BADGE_EVENTS_F');"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {170}: Create index on flow table (converted to OPTIMIZE ZORDER BY, redundant but maintained for task fidelity)\n",
    "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
    "OPTIMIZE workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow ZORDER BY (integration_id);"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# PK Violation Detection\n",
    "\n",
    "This section detects primary key violations in the flow table, logs them to the error table, and removes the erroneous records from the flow table before the final MERGE."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {180}: Insert duplicate records into E$ error table\n",
    "INSERT INTO workspace.prxbi_dw.e_wc_mercury_badgeevent_f\n",
    "(\n",
    "    ODI_PK,\n",
    "    ODI_SESS_NO,\n",
    "    ODI_ROW_ID,\n",
    "    ODI_ERR_TYPE,\n",
    "    ODI_ERR_MESS,\n",
    "    ODI_ORIGIN,\n",
    "    ODI_CHECK_DATE,\n",
    "    ODI_CONS_NAME,\n",
    "    ODI_CONS_TYPE,\n",
    "    ROW_WID,\n",
    "    INTEGRATION_ID,\n",
    "    ETL_PROC_WID,\n",
    "    EVENTEDITIONGBSCODE,\n",
    "    EVENTTYPE,\n",
    "    BADGEID,\n",
    "    SOURCE,\n",
    "    PRODUCTCODE,\n",
    "    CUSTOMERTYPE,\n",
    "    EVENTDATE,\n",
    "    CREATEDDATE,\n",
    "    BADGE_WID,\n",
    "    EVENT_EDITION_WID,\n",
    "    OBU_WID,\n",
    "    PRODUCT_WID,\n",
    "    W_UPDATE_DT,\n",
    "    W_INSERT_DT,\n",
    "    EVENT_WID\n",
    ")\n",
    "SELECT\n",
    "    uuid() AS ODI_PK, -- SYS_GUID() converted to uuid()\n",
    "    '${ODI_SESS_NO}' AS ODI_SESS_NO, -- ODI_SESS_NO parameter\n",
    "    CAST(monotonically_increasing_id() AS STRING) AS ODI_ROW_ID, -- ROWID converted\n",
    "    'F' AS ODI_ERR_TYPE,\n",
    "    'ODI-15064: The primary key PK_BADGE_EVENT is not unique.' AS ODI_ERR_MESS,\n",
    "    '(76564095)Sales and Marketing.WC_BADGE_EVENTS_F' AS ODI_ORIGIN,\n",
    "    current_timestamp() AS ODI_CHECK_DATE, -- SYSDATE converted\n",
    "    'PK_BADGE_EVENT' AS ODI_CONS_NAME,\n",
    "    'PK' AS ODI_CONS_TYPE,\n",
    "    WC_MERCURY_BADGEEVENT_F.ROW_WID,\n",
    "    WC_MERCURY_BADGEEVENT_F.INTEGRATION_ID,\n",
    "    WC_MERCURY_BADGEEVENT_F.ETL_PROC_WID,\n",
    "    WC_MERCURY_BADGEEVENT_F.EVENTEDITIONGBSCODE,\n",
    "    WC_MERCURY_BADGEEVENT_F.EVENTTYPE,\n",
    "    WC_MERCURY_BADGEEVENT_F.BADGEID,\n",
    "    WC_MERCURY_BADGEEVENT_F.SOURCE,\n",
    "    WC_MERCURY_BADGEEVENT_F.PRODUCTCODE,\
    "    WC_MERCURY_BADGEEVENT_F.CUSTOMERTYPE,\n",
    "    WC_MERCURY_BADGEEVENT_F.EVENTDATE,\n",
    "    WC_MERCURY_BADGEEVENT_F.CREATEDDATE,\n",
    "    WC_MERCURY_BADGEEVENT_F.BADGE_WID,\n",
    "    WC_MERCURY_BADGEEVENT_F.EVENT_EDITION_WID,\n",
    "    WC_MERCURY_BADGEEVENT_F.OBU_WID,\n",
    "    WC_MERCURY_BADGEEVENT_F.PRODUCT_WID,\
    "    WC_MERCURY_BADGEEVENT_F.W_UPDATE_DT,\n",
    "    WC_MERCURY_BADGEEVENT_F.W_INSERT_DT,\n",
    "    WC_MERCURY_BADGEEVENT_F.EVENT_WID\n",
    "FROM\n",
    "    workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow AS WC_MERCURY_BADGEEVENT_F\n",
    "WHERE EXISTS (\n",
    "    SELECT 1\n",
    "    FROM workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow AS SUB\n",
    "    WHERE SUB.INTEGRATION_ID = WC_MERCURY_BADGEEVENT_F.INTEGRATION_ID\n",
    "    GROUP BY SUB.INTEGRATION_ID\n",
    "    HAVING COUNT(1) > 1\n",
    ");"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {190}: Create index on E$ table (converted to OPTIMIZE ZORDER BY)\n",
    "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
    "OPTIMIZE workspace.prxbi_dw.e_wc_mercury_badgeevent_f ZORDER BY (integration_id);"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {200}: Delete duplicate records from flow table (F.2 correlated EXISTS, F.1 non-deterministic ROWID converted to MERGE DELETE)\n",
    "MERGE INTO workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow AS T\n",
    "USING (\n",
    "    SELECT DISTINCT INTEGRATION_ID\n",
    "    FROM workspace.prxbi_dw.e_wc_mercury_badgeevent_f\n",
    "    WHERE ODI_SESS_NO = '${ODI_SESS_NO}' -- Filter errors for current session\n",
    ") AS E\n",
    "ON T.INTEGRATION_ID = E.INTEGRATION_ID\n",
    "WHEN MATCHED THEN DELETE;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {210}: Insert error summary into SNP_CHECK_TAB\n",
    "INSERT INTO workspace.prxbi_dw.snp_check_tab\n",
    "(\n",
    "    SCHEMA_NAME,\n",
    "    RESOURCE_NAME,\n",
    "    FULL_RES_NAME,\n",
    "    ERR_TYPE,\n",
    "    ERR_MESS,\n",
    "    CHECK_DATE,\n",
    "    ORIGIN,\n",
    "    CONS_NAME,\n",
    "    CONS_TYPE,\n",
    "    ERR_COUNT\n",
    ")\n",
    "SELECT\n",
    "    'prxbi_dw' AS SCHEMA_NAME,\n",
    "    'wc_mercury_badgeevent_f' AS RESOURCE_NAME,\n",
    "    'workspace.prxbi_dw.wc_mercury_badgeevent_f' AS FULL_RES_NAME,\n",
    "    E.ODI_ERR_TYPE,\n",
    "    E.ODI_ERR_MESS,\n",
    "    E.ODI_CHECK_DATE,\n",
    "    E.ODI_ORIGIN,\n",
    "    E.ODI_CONS_NAME,\n",
    "    E.ODI_CONS_TYPE,\n",
    "    COUNT(1) AS ERR_COUNT\n",
    "FROM\n",
    "    workspace.prxbi_dw.e_wc_mercury_badgeevent_f AS E\n",
    "WHERE\n",
    "    E.ODI_ERR_TYPE = 'F'\n",
    "    AND E.ODI_ORIGIN = '(76564095)Sales and Marketing.WC_BADGE_EVENTS_F'\n",
    "GROUP BY\n",
    "    E.ODI_ERR_TYPE,\n",
    "    E.ODI_ERR_MESS,\n",
    "    E.ODI_CHECK_DATE,\n",
    "    E.ODI_ORIGIN,\n",
    "    E.ODI_CONS_NAME,\n",
    "    E.ODI_CONS_TYPE;"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# MERGE into Target\n",
    "\n",
    "This section performs the final MERGE operation from the cleaned flow table into the permanent target table (`WC_MERCURY_BADGEEVENT_F`).\n",
    "\n",
    "**SCEN_TASK_NO {240} and {250} are combined into a single MERGE statement (F.3, Section 7.1).**"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Inferred DDL for target table WC_MERCURY_BADGEEVENT_F\n",
    "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_mercury_badgeevent_f\n",
    "(\n",
    "    ROW_WID             BIGINT GENERATED ALWAYS AS IDENTITY,\n",
    "    INTEGRATION_ID      STRING NOT NULL,\n",
    "    ETL_PROC_WID        BIGINT,\n",
    "    EVENTEDITIONGBSCODE STRING,\n",
    "    EVENTTYPE           STRING,\n",
    "    BADGEID             STRING,\n",
    "    SOURCE              STRING,\n",
    "    PRODUCTCODE         STRING,\
    "    CUSTOMERTYPE        STRING,\n",
    "    EVENTDATE           STRING,\n",
    "    CREATEDDATE         STRING,\n",
    "    BADGE_WID           BIGINT,\n",
    "    EVENT_EDITION_WID   BIGINT,\n",
    "    OBU_WID             BIGINT,\n",
    "    PRODUCT_WID         BIGINT,\
    "    EVENT_WID           BIGINT,\
    "    W_UPDATE_DT         TIMESTAMP,\n",
    "    W_INSERT_DT         TIMESTAMP\n",
    ")\n",
    "USING DELTA\n",
    "TBLPROPERTIES (\"delta.enableChangeDataFeed\" = \"true\");"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {240} (UPDATE) and {250} (INSERT) combined into a single MERGE statement (F.3, Section 7.1)\n",
    "MERGE INTO workspace.prxbi_dw.wc_mercury_badgeevent_f AS T\n",
    "USING workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow AS S\n",
    "ON T.INTEGRATION_ID = S.INTEGRATION_ID\n",
    "WHEN MATCHED THEN UPDATE SET\n",
    "    T.EVENTEDITIONGBSCODE   = S.EVENTEDITIONGBSCODE,\n",
    "    T.EVENTTYPE             = S.EVENTTYPE,\n",
    "    T.BADGEID               = S.BADGEID,\n",
    "    T.SOURCE                = S.SOURCE,\n",
    "    T.PRODUCTCODE           = S.PRODUCTCODE,\
    "    T.CUSTOMERTYPE          = S.CUSTOMERTYPE,\n",
    "    T.EVENTDATE             = S.EVENTDATE,\n",
    "    T.CREATEDDATE           = S.CREATEDDATE,\n",
    "    T.BADGE_WID             = S.BADGE_WID,\n",
    "    T.EVENT_EDITION_WID     = S.EVENT_EDITION_WID,\n",
    "    T.OBU_WID               = S.OBU_WID,\n",
    "    T.PRODUCT_WID           = S.PRODUCT_WID,\
    "    T.EVENT_WID             = S.EVENT_WID,\n",
    "    T.ETL_PROC_WID          = ${ETL_PROC_WID},\n",
    "    T.W_UPDATE_DT           = current_timestamp()\n",
    "WHEN NOT MATCHED THEN INSERT (\n",
    "    INTEGRATION_ID,\n",
    "    EVENTEDITIONGBSCODE,\n",
    "    EVENTTYPE,\n",
    "    BADGEID,\n",
    "    SOURCE,\n",
    "    PRODUCTCODE,\
    "    CUSTOMERTYPE,\n",
    "    EVENTDATE,\n",
    "    CREATEDDATE,\n",
    "    BADGE_WID,\n",
    "    EVENT_EDITION_WID,\n",
    "    OBU_WID,\n",
    "    PRODUCT_WID,\
    "    EVENT_WID,\n",
    "    ETL_PROC_WID,\n",
    "    W_UPDATE_DT,\n",
    "    W_INSERT_DT\n",
    ") VALUES (\n",
    "    S.INTEGRATION_ID,\n",
    "    S.EVENTEDITIONGBSCODE,\n",
    "    S.EVENTTYPE,\n",
    "    S.BADGEID,\n",
    "    S.SOURCE,\n",
    "    S.PRODUCTCODE,\
    "    S.CUSTOMERTYPE,\n",
    "    S.EVENTDATE,\n",
    "    S.CREATEDDATE,\n",
    "    S.BADGE_WID,\n",
    "    S.EVENT_EDITION_WID,\n",
    "    S.OBU_WID,\n",
    "    S.PRODUCT_WID,\n",
    "    S.EVENT_WID,\n",
    "    ${ETL_PROC_WID},\n",
    "    current_timestamp(),\n",
    "    current_timestamp()\n",
    ");"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Optimize Target\n",
    "\n",
    "Optimizes the target table after the MERGE operation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Optimize target table (replaces index creation/stats gathering)\n",
    "SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;\n",
    "OPTIMIZE workspace.prxbi_dw.wc_mercury_badgeevent_f ZORDER BY (integration_id);"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Cleanup\n",
    "\n",
    "This section removes temporary flow and staging tables."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {270}: Drop flow table\n",
    "DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_mercury_badgeevent_f_flow;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- SCEN_TASK_NO {300}: Drop staging table\n",
    "DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_event_stg;"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Validation\n",
    "\n",
    "This section provides queries to validate the results of the ETL process, including final record counts and error summaries."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Final record count in the target table\n",
    "SELECT COUNT(*) AS final_target_record_count FROM workspace.prxbi_dw.wc_mercury_badgeevent_f;"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "-- Error summary from SNP_CHECK_TAB for this process\n",
    "SELECT * FROM workspace.prxbi_dw.snp_check_tab WHERE schema_name = 'prxbi_dw' AND origin = '(76564095)Sales and Marketing.WC_BADGE_EVENTS_F';"
   ]
  },
  {
   "cell_type": "markdown",
   "source": [
    "# Conversion Notes and Manual Actions Required\n",
    "\n",
    "- **Assumed DDL:** DDL for `workspace.prxbi_dw.wc_etl_parameters` and other dimension tables (`wc_badge_details_d`, `wc_event_ed_d`, `wc_badge_product_d`, `wc_event_d`, `wc_mercury_badgeevent_ts`) were assumed based on their usage in the ODI source. These tables must exist in Databricks with compatible schemas.\n",
    "- **Identity Column:** The `ROW_WID` in the target table `wc_mercury_badgeevent_f` is assumed to be `GENERATED ALWAYS AS IDENTITY` based on the ODI `SEQUENCE.NEXTVAL` pattern. It has been excluded from explicit `INSERT` and `UPDATE` column lists in the MERGE statement (F.4).\n",
    "- **ODI MAX Self-Join Deduplication (F.12):** The complex ODI pattern for deduplication in `SCEN_TASK_NO {50}` has been rewritten using `ROW_NUMBER()` over the partition key (`EVENTEDITIONGBSCODE`, `EVENTTYPE`, `BADGEID`, `SOURCE`, `PRODUCTCODE`, `CUSTOMERTYPE`) ordered by `INT_INSERT_DATE DESC` to ensure deterministic single-row selection per key.\n",
    "- **Non-deterministic functions in DELETE/MERGE (F.1, F.2):** The use of `ROWID` and correlated `EXISTS` subqueries in `DELETE` statements has been replaced with a `MERGE DELETE` on `INTEGRATION_ID` to ensure deterministic behavior and compatibility with Delta Lake.\n",
    "- **Oracle Tuple-SET UPDATE (F.3):** The Oracle-specific `UPDATE ... SET (col1, col2) = (SELECT ...)` syntax in `SCEN_TASK_NO {240}` has been combined with `SCEN_TASK_NO {250}` into a single `MERGE INTO ... WHEN MATCHED THEN UPDATE SET ... WHEN NOT MATCHED THEN INSERT ...` statement.\n",
    "- **Parameter Handling:** ODI `#GLOBAL.*` and `#SCHEMA.*` parameters are converted to Databricks widgets (e.g., `dbutils.widgets.text` in Python, `${PARAM_NAME}` in SQL).\n",
    "- **Schema and Table Naming:** All Oracle schema and table names have been converted to lowercase and prefixed with `workspace.`, and environment suffixes stripped (e.g., `PRXBI_DW_SEP.C$_...` becomes `workspace.prxbi_dw.c_badge_event_stg`).\n",
    "- **Date/Timestamp Formats (F.8):** Oracle `TO_TIMESTAMP` and `TO_DATE` format strings have been converted to their Spark SQL equivalents.\n",
    "- **Optimize ZORDER Stats Check (F.11):** Every `OPTIMIZE ... ZORDER BY` statement is preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` in the same cell.\n",
    "- **`SNP_CHECK_TAB` Origin:** The origin string `'(76564095)Sales and Marketing.WC_BADGE_EVENTS_F'` is preserved as a literal string in the `snp_check_tab` inserts and deletes, as it represents a unique identifier from the ODI context."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "python",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.18"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
```